# A/B Test Prompt Versions and Ship the Winner

Version a prompt, optimize it automatically, run a structured comparison, and promote the winner to production, all without changing application code.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/full-prompt-lifecycle.ipynb)

| Time | Difficulty |
|------|------------|
| 30 min | Intermediate |

You have an LLM-powered assistant in production. The system prompt is too vague, so the model guesses when it should be precise. You need to measure the current version, generate an improved candidate, compare them head-to-head, and promote the winner, all without touching your application code.

By the end, you will have a repeatable, evidence-based process for iterating on any prompt in production.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- OpenAI API key (`OPENAI_API_KEY`)
- Python 3.9+

## Install

In [1]:
# Packages installed via install.ipynb


In [2]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["FI_BASE_URL"] = "http://localhost:8000"
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


## Step 1: Store your current prompt as v1

Move your system prompt into FutureAGI Prompt Management so you can version, optimize, and swap it without redeploying. For this walkthrough, the starting prompt is a one-line system message for an HR onboarding assistant.

In [3]:
import os
from fi.prompt import Prompt
from fi.prompt.types import PromptTemplate, SystemMessage, UserMessage, ModelConfig

SYSTEM_PROMPT = "You are an HR assistant. Help new employees with onboarding questions."

prompt_client = Prompt(
    template=PromptTemplate(
        name="hr-onboarding",
        messages=[
            SystemMessage(content=SYSTEM_PROMPT),
            UserMessage(content="{{employee_message}}"),
        ],
        model_configuration=ModelConfig(
            model_name="gpt-4o-mini",
            temperature=0.7,
            max_tokens=500,
        ),
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

prompt_client.create()
prompt_client.commit_current_version(
    message="v1: bare-bones HR prompt, no policy details",
    label="production",
)

print(f"Created: {prompt_client.template.name} ({prompt_client.template.version})")
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


/Users/karthikavinash/Desktop/repos/futureagi-sdk/python/fi/api/types.py:15: UserWarning: Field name "json" in "RequestConfig" shadows an attribute in parent "BaseModel"
  class RequestConfig(BaseModel):


  fi.utils.logging | WARNING | Template not found in the backend. Create a new template before running.
  fi.utils.logging | INFO | Fetching template version history for hr-onboarding
Created: hr-onboarding (v1)


Your application fetches this prompt at runtime by label. When you promote a new version later, every instance picks it up automatically.

In [4]:
import os
from fi.prompt import Prompt


def get_system_prompt() -> str:
    prompt = Prompt.get_template_by_name(
        name="hr-onboarding",
        label="production",
        fi_api_key=os.environ["FI_API_KEY"],
        fi_secret_key=os.environ["FI_SECRET_KEY"],
    )
    return prompt.template.messages[0].content
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


See [Prompt Versioning](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-versioning) for `compile()`, rollback, and label management.

## Step 2: Measure the baseline so you know what to beat

Before optimizing, measure how v1 actually performs. Build a test dataset with realistic questions and run evals to establish a baseline.

In [6]:
import os
import litellm
from fi.prompt import Prompt
from fi.evals import evaluate

test_dataset = [
    {
        "question": "I start next Monday. Which health plan should I pick if I want dental and vision included?",
        "context": "The company offers Basic (100% employer-paid, $500 deductible), Plus ($45/mo, $250 deductible, includes vision), and Premium ($120/mo, $0 deductible, includes dental + vision + mental health). Enrollment window closes 30 days after start date.",
    },
    {
        "question": "I'm a remote employee starting in April. How do I get my laptop and dev tools?",
        "context": "Remote employees: laptop ships to home address 5 business days before start date. VPN via Cisco AnyConnect, credentials in welcome email. Tools: Slack, Jira, GitHub, Figma (eng/design). IT support: it-help@company.com.",
    },
    {
        "question": "I'm joining as a contractor from Sweden. Do I get PTO?",
        "context": "Contractors do not receive PTO. They set their own schedules per SOW. International contractors must comply with local labor laws. No visa sponsorship for contractors.",
    },
    {
        "question": "How much PTO do I get as a full-time employee, and can I use it during my first month?",
        "context": "Full-time: 20 days PTO, 10 sick days. Accrual: 1.67 days/month after 90-day probation. Sick days available immediately. 5 unused days roll over.",
    },
    {
        "question": "My start date is March 24 and I haven't gotten my welcome email. What should I do?",
        "context": "Welcome emails sent 7 business days before start date by People Ops (people-ops@company.com). Contains VPN credentials, Slack invite, benefits link, Day 1 schedule.",
    },
    {
        "question": "I'm hybrid. Do I need a badge for the office? And where do I pick up my laptop?",
        "context": "Hybrid employees: laptop ships home or pick up on-site (coordinate with IT). Badge required for on-site days from Security desk in lobby with government ID. VPN for remote days.",
    },
]

prompt = Prompt.get_template_by_name(
    name="hr-onboarding",
    label="production",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print(f"{'Question':<60} {'Complete':>10} {'Relevant':>10}")
print("-" * 82)

for item in test_dataset:
    messages = prompt.compile(employee_message=item["question"])

    response = litellm.completion(
        model="gpt-4o-mini",
        messages=messages,
    )
    output = response.choices[0].message.content

    completeness_result = evaluate(
        "completeness",
        input=item["question"],
        output=output,
        model="turing_flash",
    )

    relevance_result = evaluate(
        "context_relevance",
        input=item["question"],
        context=item["context"],
        model="turing_flash",
    )

    print(f"{item['question'][:58]:<60} {str(completeness_result.score):>10} {str(relevance_result.score):>10}")
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


Question                                                       Complete   Relevant
----------------------------------------------------------------------------------
I start next Monday. Which health plan should I pick if I           0.4        1.0
I'm a remote employee starting in April. How do I get my l          1.0        0.8
I'm joining as a contractor from Sweden. Do I get PTO?              0.6        1.0
How much PTO do I get as a full-time employee, and can I u          1.0        1.0
My start date is March 24 and I haven't gotten my welcome           1.0        0.6
I'm hybrid. Do I need a badge for the office? And where do          1.0        1.0


With a one-line system prompt, expect low completeness on questions that need specific policy details. The model will give generic answers when it should cite enrollment deadlines, the 90-day probation period, or contractor-vs-full-time distinctions. Those gaps are what the optimizer will fix.

## Step 3: Generate an improved prompt automatically

Instead of manually rewriting the prompt, let the optimizer do it. `MetaPromptOptimizer` uses a teacher LLM to iteratively analyze what is wrong with the current prompt's outputs and rewrites it based on eval scores.

In [7]:
import os
from fi.opt.base import Evaluator
from fi.opt.generators import LiteLLMGenerator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.optimizers import MetaPromptOptimizer

baseline_prompt = (
    "You are an HR assistant. Help new employees with onboarding questions.\n\n"
    "Employee question: {question}"
)

optimization_dataset = [
    {
        "question": "I start next Monday. Which health plan should I pick if I want dental and vision included?",
        "target_response": "For dental and vision coverage, you'll want either the Plus plan ($45/mo, $250 deductible, includes vision) or the Premium plan ($120/mo, $0 deductible, includes dental + vision + mental health). If you only need vision, Plus is the cost-effective choice. If you want the full package with mental health coverage, go Premium. Important: your enrollment window closes 30 days after your start date.",
    },
    {
        "question": "I'm a remote employee starting in April. How do I get my laptop and dev tools?",
        "target_response": "Your laptop will ship to your home address 5 business days before your start date. You'll get Cisco AnyConnect VPN credentials in your welcome email. For dev tools, you'll have access to Slack, Jira, and GitHub on Day 1. If you're in Engineering or Design, Figma is included too. If anything doesn't arrive on time, reach out to it-help@company.com.",
    },
    {
        "question": "I'm joining as a contractor from Sweden. Do I get PTO?",
        "target_response": "Contractors don't receive PTO. You set your own schedule per your Statement of Work. As an international contractor, you'll also want to make sure you're complying with Swedish labor laws on your end. If you have questions about your SOW terms, reach out to People Ops at people-ops@company.com.",
    },
    {
        "question": "How much PTO do I get as a full-time employee, and can I use it during my first month?",
        "target_response": "Full-time employees get 20 days of PTO and 10 sick days per year. PTO accrues at 1.67 days per month, but it's only available after your 90-day probation period, so you won't be able to use PTO in your first month. Sick days are available immediately.",
    },
    {
        "question": "My start date is March 24 and I haven't received a welcome email yet.",
        "target_response": "Welcome emails go out 7 business days before your start date, so you should have received it by now. Contact People Ops directly at people-ops@company.com. The email contains your VPN credentials, Slack invite, benefits enrollment link, and Day 1 schedule.",
    },
    {
        "question": "I'm hybrid. Do I need a badge for the office? And where do I pick up my laptop?",
        "target_response": "Yes, you'll need a badge for on-site days. Pick it up from the Security desk in the lobby on your first day in the office (bring a government-issued ID). For your laptop, hybrid employees can either have it shipped home or pick it up on-site. Coordinate with IT at it-help@company.com.",
    },
]

evaluator = Evaluator(
    eval_template="completeness",
    eval_model_name="turing_flash",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

data_mapper = BasicDataMapper(
    key_map={
        "input": "question",
        "output": "generated_output",
    }
)

teacher = LiteLLMGenerator(model="gpt-4o", prompt_template="{prompt}")

optimizer = MetaPromptOptimizer(
    teacher_generator=teacher,
)

result = optimizer.optimize(
    evaluator=evaluator,
    data_mapper=data_mapper,
    dataset=optimization_dataset,
    initial_prompts=[baseline_prompt],
    task_description="Answer HR onboarding questions for new hires. "
                     "Responses should reference specific company policies (benefits plans, PTO accrual, "
                     "IT provisioning), distinguish between full-time employees and contractors, handle "
                     "international hire edge cases, and maintain a warm but precise tone.",
    eval_subset_size=6,
)

print(f"Baseline score:  {result.history[0].average_score:.3f}")
print(f"Optimized score: {result.final_score:.3f}")
print(f"\nBest prompt found:")
print("-" * 60)
print(result.best_generator.get_prompt_template())
print("-" * 60)

print("\nOptimization history:")
for i, iteration in enumerate(result.history):
    print(f"  Round {i+1}: score={iteration.average_score:.3f}")
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


/opt/miniconda3/envs/cb/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baseline score:  0.700
Optimized score: 1.000

Best prompt found:
------------------------------------------------------------
You are an HR assistant specializing in onboarding new employees. Your goal is to address any onboarding questions with comprehensive, precise, and customized responses. Follow these guidelines when crafting your answers:

1. **Comprehensive and Specific Responses**: Provide answers that cover all aspects of the question, both explicit and implied. Reference specific company policies on benefits, PTO accrual, IT provisioning, and other relevant areas. Include necessary timelines and procedures.

2. **Status Differentiation**: Clearly distinguish the entitlements and responsibilities of full-time employees versus contractors. Ensure that your responses address the specific employment status of the individual.

3. **Edge Case Handling**: For international hires, hybrid or remote employees, ensure your response includes relevant jurisdiction-specific guidance, cul

Optimization typically takes 2-5 minutes. You should see clear score improvement across rounds as the optimizer adds specific policy instructions, contractor handling, and escalation rules that the vague v1 was missing.

See [Prompt Optimization](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization) for alternative optimizers like Bayesian Search and ProTeGi.

## Step 4: Save the optimized prompt as v2 (without promoting it yet)

Take the optimizer's output and version it as v2. Do not promote it yet. You will A/B test it first.

The optimizer outputs a prompt tailored to the failure patterns it found in your dataset. Use `result.best_generator.get_prompt_template()` from the previous step as `OPTIMIZED_PROMPT` below.

In [8]:
from fi.prompt.types import PromptTemplate, SystemMessage, UserMessage, ModelConfig

# Use your optimizer output here:
OPTIMIZED_PROMPT = result.best_generator.get_prompt_template()

prompt_client.create_new_version(
    template=PromptTemplate(
        name="hr-onboarding",
        messages=[
            SystemMessage(content=OPTIMIZED_PROMPT),
            UserMessage(content="{{employee_message}}"),
        ],
        model_configuration=ModelConfig(
            model_name="gpt-4o-mini",
            temperature=0.5,
            max_tokens=500,
        ),
    ),
)

prompt_client.save_current_draft()
prompt_client.commit_current_version(
    message="v2: optimized via MetaPrompt, adds policy details and contractor handling",
)

print(f"v2 created: {prompt_client.template.version}")
print("Not yet promoted to production.")

  fi.utils.logging | INFO | Fetching template version history for hr-onboarding
  fi.utils.logging | INFO | Fetching template version history for hr-onboarding
  fi.utils.logging | INFO | Fetching template version history for hr-onboarding
v2 created: v2
Not yet promoted to production.


## Step 5: Run a structured A/B test between v1 and v2

You have two versions. Instead of eyeballing outputs, run a structured comparison in the FutureAGI UI: same dataset, two prompt variants, eval scores, and a clear winner.

**Prepare the dataset:**

Save the following as `onboarding-test.csv`.

```csv
question,context,expected_answer
"Which health plan includes dental and vision?","Three plans: Basic (100% employer-paid, $500 deductible), Plus ($45/mo, $250 deductible, vision), Premium ($120/mo, $0 deductible, dental + vision + mental health). Enrollment closes 30 days after start date.","Premium includes both dental and vision. Plus includes vision only. Mention the 30-day enrollment deadline."
"I'm a remote engineer starting April 1. How do I get my laptop?","Remote employees: laptop ships to home address 5 business days before start date. VPN via Cisco AnyConnect. Tools: Slack, Jira, GitHub, Figma (eng/design). IT support: it-help@company.com.","Laptop ships 5 business days before start. VPN credentials in welcome email. Mention Figma access for engineering."
"I'm a contractor from Sweden. What benefits do I get?","Contractors do not receive company benefits or PTO. They operate under their SOW. International contractors must comply with local labor laws. No visa sponsorship for contractors.","Contractors don't get company benefits. Refer to SOW. Mention local labor law compliance."
"Can I use PTO in my first month as a full-time employee?","Full-time: 20 days PTO, 10 sick days. Accrual: 1.67 days/month after 90-day probation. Sick days available immediately. 5 unused days roll over.","No. PTO is available after the 90-day probation period. Sick days are available immediately."
"I'm hybrid and need to know about badge access and laptop pickup.","Hybrid: laptop ships home or pick up on-site (coordinate with IT). Badge required for on-site days from Security desk in lobby with government ID. VPN for remote days.","Badge from Security desk on Day 1 with ID. Laptop: choose shipping or on-site pickup. Mention VPN for remote days."
"My start date is March 24 and I haven't gotten my welcome email. What should I do?","Welcome emails sent 7 business days before start date by People Ops (people-ops@company.com). Contains VPN credentials, Slack invite, benefits link, Day 1 schedule.","Contact People Ops at people-ops@company.com. Explain what the welcome email contains."
```

**Set up the experiment:**

Go to [app.futureagi.com](https://app.futureagi.com) -> **Dataset** -> **Add Dataset** -> upload `onboarding-test.csv`. Open the dataset, click **Experiment**, name it `v1-vs-v2-onboarding`, and set the baseline column to `expected_answer`. Configure two prompt templates: one with the v1 system message and one with the v2 optimized message (same user message format: `Context: {{context}}\nEmployee question: {{question}}`). Click **Run**, then evaluate with `completeness` and `groundedness`.

See [Experimentation](https://docs.futureagi.com/docs/cookbook/quickstart/experimentation-compare-prompts) for the full experiment setup walkthrough with screenshots.

Expect v2 to outperform v1 across the board. The contractor question is the most telling: v1 will likely produce a generic answer, while v2 correctly states that contractors operate under their SOW. The PTO question is another clear gap: v1 will not mention the 90-day probation period, but v2 will. Once you confirm v2 wins, move to the next step.

## Step 6: Promote the winner to production

The A/B test confirmed v2 is better. Promote it to production. Every agent instance calling `get_template_by_name(label="production")` picks it up on the next request.

In [9]:
import os
from fi.prompt import Prompt

Prompt.assign_label_to_template_version(
    template_name="hr-onboarding",
    version="v2",
    label="production",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print("v2 is now the production prompt.")
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


v2 is now the production prompt.


Your `get_system_prompt()` function from Step 1 now serves v2 automatically. No code change, no redeploy.

## Step 7: Roll back if something goes wrong

If v2 causes unexpected issues, revert in one line:

In [10]:
import os
from fi.prompt import Prompt

Prompt.assign_label_to_template_version(
    template_name="hr-onboarding",
    version="v1",
    label="production",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print("Rolled back to v1.")
os.environ["OPENAI_API_KEY"] = "sk-proj-your-openai-key"


Rolled back to v1.


Promotion and rollback are label reassignments, not deployments. If v2 causes an issue, you can revert in seconds and investigate at your own pace.

You can also check the full version timeline:

In [11]:
versions = prompt_client.list_template_versions()

for v in versions:
    draft = "draft" if v.get("isDraft") else "committed"
    print(f"  {v['templateVersion']}  {draft}  {v['createdAt']}")

  fi.utils.logging | INFO | Fetching template version history for hr-onboarding
  v2  committed  2026-04-01T05:14:34.433100Z
  v1  committed  2026-04-01T03:42:37.918018Z


Every version is immutable. As your prompt evolves (v3 adds parental leave policy, v4 adds a new office location), this history becomes your changelog. Run the same cycle each time: optimize, test, promote.

## What you solved

You improved a production prompt through a structured cycle (version, optimize, A/B test, promote) without changing any application code. And you can roll back in one line if anything goes wrong.

- **"How bad is my current prompt?"** Baseline evaluation with `completeness` and `context_relevance` showed where v1 fell short.
- **"How do I improve it without guessing?"** `MetaPromptOptimizer` generated a better prompt automatically, guided by eval scores on your dataset.
- **"Which version is actually better?"** A structured experiment compared both versions on the same dataset with weighted metrics.
- **"How do I ship the winner safely?"** A single label change promoted v2 to production. Another label change rolls it back.

## Explore further

- [Prompt Versioning](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-versioning): Labels, rollback, compile(), version history
- [Prompt Optimization](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization): MetaPrompt, Bayesian Search, and more
- [Experimentation](https://docs.futureagi.com/docs/cookbook/quickstart/experimentation-compare-prompts): Multi-model A/B tests with weighted scoring
- [Running Your First Eval](https://docs.futureagi.com/docs/cookbook/quickstart/first-eval): Core eval patterns and 72+ built-in metrics